In [3]:
# 1. 먼저 pyreadstat 패키지 설치
!pip install pyreadstat

# 2. 구글 드라이브 연동
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import pyreadstat

# 3. 경로 설정
DATA_DIR = '/content/drive/MyDrive/hdac_data'

Mounted at /content/drive


In [4]:
import glob

# 구글 드라이브 전체에서 파일 위치 찾기
found_files = glob.glob('/content/drive/**/*.sas7bdat', recursive=True)
print("🔍 파일이 실제로 존재하는 경로:", found_files)

🔍 파일이 실제로 존재하는 경로: ['/content/drive/MyDrive/MyDrive hdac_data /hn24_all.sas7bdat']


In [8]:
DATA_DIR = '/content/drive/MyDrive/hdac_data'

target_cols = ['age', 'sex', 'kstrata', 'psu', 'wt_ntr']
df, meta = pyreadstat.read_sas7bdat(f'{DATA_DIR}/hn24_all.sas7bdat', usecols=target_cols)

print("전체 데이터 크기 (행, 열):", df.shape)

전체 데이터 크기 (행, 열): (6997, 5)


In [16]:
import pandas as pd
import pyreadstat

# 1. 파일 경로 설정
DATA_DIR = '/content/drive/MyDrive/hdac_data'

# 2. 파일 내에 실제로 존재하는 모든 열 이름(대소문자 포함)을 먼저 읽어와 확인합니다.
_, meta_all = pyreadstat.read_sas7bdat(f'{DATA_DIR}/hn24_all.sas7bdat', metadataonly=True)
available_cols = meta_all.column_names

# 3. 우리가 불러오고자 하는 타겟 변수 목록
target_cols = [
    'age', 'sex', 'kstrata', 'psu', 'wt_ntr',
    'genertn',    # 1인 가구 여부
    'HE_khei',    # 식생활평가지수
    'incm',       # 소득
    'edu',        # 교육수준
    'BD1_11',     # 음주
    'pa_aerobic'  # 유산소 운동
]

# 실제 파일에 존재하는 변수들만 골라냅니다 (대소문자 구문 문제 대응)
valid_target_cols = []
missing_cols = []

for col in target_cols:
    # 대소문자 구분 없이 매칭되는 변수가 있는지 확인
    matched = [c for c in available_cols if c.lower() == col.lower()]
    if matched:
        valid_target_cols.append(matched[0])
    else:
        missing_cols.append(col)

print("✅ 불러올 유효한 변수들:", valid_target_cols)
if missing_cols:
    print("⚠️ 데이터에 없는 변수들:", missing_cols)

# 4. 유효한 변수들만 골라서 데이터 로드
df, meta = pyreadstat.read_sas7bdat(f'{DATA_DIR}/hn24_all.sas7bdat', usecols=valid_target_cols)
print("확장된 데이터 크기 (행, 열):", df.shape)

# 5. 대상 선정 단계별 인원 확인
print("=" * 40)
print(" [대상 선정 단계별 인원 확인]")
print("=" * 40)
print(f"1단계 - 초기 전체 데이터 수: {len(df):,}명")

if 'wt_ntr' in df.columns:
    df_filtered = df.dropna(subset=['wt_ntr'])
    print(f"2단계 - 영양조사 가중치(wt_ntr) 결측 제외 후: {len(df_filtered):,}명")
else:
    df_filtered = df
    print("2단계 - 가중치 변수 없음 (전체 유지)")

# 6. 주요 변수 원 코드 빈도표 확인
print("\n" + "=" * 40)
print(" [주요 변수 원 코드 및 빈도 검증]")
print("=" * 40)

if 'genertn' in df_filtered.columns:
    print("--- 1인 가구 여부(genertn) 원 코드 빈도표 ---")
    print(df_filtered['genertn'].value_counts(dropna=False))

if 'HE_khei' in df_filtered.columns:
    print("\n--- 식생활평가지수(HE_khei) 요약 통계량 ---")
    print(df_filtered['HE_khei'].describe())

✅ 불러올 유효한 변수들: ['age', 'sex', 'kstrata', 'psu', 'wt_ntr', 'genertn', 'incm', 'edu', 'BD1_11', 'pa_aerobic']
⚠️ 데이터에 없는 변수들: ['HE_khei']
확장된 데이터 크기 (행, 열): (6997, 10)
 [대상 선정 단계별 인원 확인]
1단계 - 초기 전체 데이터 수: 6,997명
2단계 - 영양조사 가중치(wt_ntr) 결측 제외 후: 6,805명

 [주요 변수 원 코드 및 빈도 검증]
--- 1인 가구 여부(genertn) 원 코드 빈도표 ---
genertn
4.0    2807
2.0    1622
1.0    1056
5.0     625
7.0     323
6.0     278
3.0      85
9.0       9
Name: count, dtype: int64
